In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader,Subset,Dataset,TensorDataset

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import os
import cv2
import copy
from skimage.util import random_noise
from sklearn.model_selection import train_test_split
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from mpl_toolkits.axes_grid1 import ImageGrid
from PIL import Image

seed = 4912
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [9]:
### START CODE HERE ###

class DownSamplingBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(DownSamplingBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        x = self.pool(x)
        return x

class UpSamplingBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(UpSamplingBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding)
        self.relu = nn.ReLU()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        x = self.upsample(x)
        return x

class Autoencoder(nn.Module):
    def __init__(self, architecture=[64, 128, 256, 512]):
        super().__init__()
        self.conv_in = nn.Conv2d(3, architecture[0], kernel_size=3, stride=1, padding=1)
        self.down_blocks = nn.ModuleList()
        in_ch = architecture[0]
        for out_ch in architecture[1:]:
            self.down_blocks.append(DownSamplingBlock(in_ch, out_ch))
            in_ch = out_ch
        self.up_blocks = nn.ModuleList()
        rev_arch = architecture[::-1]
        for i in range(len(rev_arch)-1):
            self.up_blocks.append(UpSamplingBlock(rev_arch[i], rev_arch[i+1]))
        self.conv_out = nn.Conv2d(architecture[0], 3, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        x = self.conv_in(x)

        for down in self.down_blocks:
            x = down(x)
        for up in self.up_blocks:
            x = up(x)

        x = self.conv_out(x)
        return x
    
### END CODE HERE ###

In [ ]:
model = Autoencoder(architecture=[64, 128, 256])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.load_state_dict(torch.load('../autoencoder_grid_search.pth', map_location=device))
model.to(device)
model.eval()

os.makedirs('./outputs', exist_ok=True)

with torch.no_grad():
    files = os.listdir('./test')
    files = [os.path.join('./test', file) for file in files]

    for file in files:
        img = Image.open(file).convert('RGB').resize((128, 128))
        tensor_img = transforms.ToTensor()(img).unsqueeze(0).to(device)

        outputs = model(tensor_img)
        out_img = outputs[0].cpu()

        out_img_np = out_img.permute(1, 2, 0).numpy()
        out_img_np = (out_img_np * 255).clip(0, 255).astype('uint8')
        out_img_pil = Image.fromarray(out_img_np)

        base_name = os.path.basename(file)
        out_img_pil.save(f'./outputs/{base_name}', format='JPEG')
